# 文件信息提取

In [1]:
import os
import re
import pandas as pd
from datetime import datetime

Data_path = r"E:\\Grad\\data\\中广核响泉风电场"
# Data_path="..//data//中广核响泉风电场"

folder_names = [
    "中广核响泉风电场04#发电机轴承更换后正常数据",
    "中广核响泉风电场04#发电机轴承故障数据",
    "中广核响泉风电场04#主轴承更换后正常数据",
    "中广核响泉风电场04#主轴承故障数据",
    "中广核响泉风电场06#发电机轴承更换后正常数据",
    "中广核响泉风电场06#发电机轴承故障数据",
    "中广核响泉风电场12#发电机轴承更换后正常数据",
    "中广核响泉风电场12#发电机轴承故障数据",
    "中广核响泉风电场14#发电机轴承更换后正常数据",
    "中广核响泉风电场14#发电机轴承故障数据",
    "中广核响泉风电场16#发电机轴承更换后正常数据",
    "中广核响泉风电场16#发电机轴承故障数据",
    "中广核响泉风电场18#发电机轴承更换后数据",
    "中广核响泉风电场18#发电机轴承故障数据",
    "中广核响泉风电场19#发电机轴承更换后数据",
    "中广核响泉风电场19#发电机轴承故障数据",
]

class WindFarmDataReader:
    def __init__(self, base_path):
        self.base_path = base_path
        
    def parse_folder_name_1(self, folder_name):
        """
        解析一级文件夹名称，提取关键信息
        """
        info = {'folder_name': folder_name}  # 保留原始文件夹名
        
        # 提取机组编号
        unit_match = re.search(r'(\d+)#', folder_name)
        info['unit_no'] = unit_match.group(1) if unit_match else '未知'
        
        # 提取部件
        component_match = re.search(r'(发电机轴承|主轴承|齿轮箱)', folder_name)
        info['component'] = component_match.group(1) if component_match else '未知'
        
        # 提取更详细的部件信息（针对特殊情况）
        if '一级行星轮系' in folder_name:
            info['component_detail'] = '一级行星轮系'
        else:
            info['component_detail'] = '无'
        
        # 提取状态
        if '故障' in folder_name:
            info['status'] = '故障'
        elif '正常' in folder_name:
            info['status'] = '正常'
        else:
            info['status'] = '未知'
        
        # 判断是否更换后
        info['after_replacement'] = '更换后' in folder_name
        
        # 构建完整路径
        info['full_path'] = os.path.join(self.base_path, folder_name)
        
        # 检查文件夹是否存在
        info['exists'] = os.path.exists(info['full_path']) and os.path.isdir(info['full_path'])
        
        return info
    
    def create_dataframe(self, folder_names_list):
        """
        将文件夹名称列表转换为DataFrame
        """
        # 解析所有文件夹
        parsed_data = []
        for folder_name in folder_names_list:
            folder_info = self.parse_folder_name_1(folder_name)
            parsed_data.append(folder_info)
        
        # 创建DataFrame
        df = pd.DataFrame(parsed_data)
        
        # 重新排列列的顺序
        column_order = ['folder_name', 'unit_no', 'component', 'component_detail', 
                       'status', 'after_replacement',  ]
        df = df[column_order]
        
        return df
    

Data=WindFarmDataReader(Data_path)
# 创建一级文件夹的DataFrame
Data_info=Data.create_dataframe(folder_names)

Data_info

,folder_name,unit_no,component,component_detail,status,after_replacement
0,中广核响泉风电场04#发电机轴承更换后正常数据,04,发电机轴承,无,正常,True
1,中广核响泉风电场04#发电机轴承故障数据,04,发电机轴承,无,故障,False
2,中广核响泉风电场04#主轴承更换后正常数据,04,主轴承,无,正常,True
3,中广核响泉风电场04#主轴承故障数据,04,主轴承,无,故障,False
4,中广核响泉风电场06#发电机轴承更换后正常数据,06,发电机轴承,无,正常,True
5,中广核响泉风电场06#发电机轴承故障数据,06,发电机轴承,无,故障,False
6,中广核响泉风电场12#发电机轴承更换后正常数据,12,发电机轴承,无,正常,True
7,中广核响泉风电场12#发电机轴承故障数据,12,发电机轴承,无,故障,False
8,中广核响泉风电场14#发电机轴承更换后正常数据,14,发电机轴承,无,正常,True
9,中广核响泉风电场14#发电机轴承故障数据,14,发电机轴承,无,故障,False


# 定义提取数据函数

In [2]:
import os
import re
import pandas as pd
import numpy as np

# ==========================================
# 1. 辅助函数：标签分类与清洗逻辑
# ==========================================

def clean_component_label(comp_str):
    """
    将 component_detail 转换为标准工程标签
    例如：'非驱动端径向' -> 'NDE_径向'
    """
    if pd.isna(comp_str):
        return "Unknown"

    s = str(comp_str)

    # 定义映射规则
    if "非驱动端" in s:
        prefix = "NDE" # Non-Drive End
    elif "驱动端" in s:
        prefix = "DE"  # Drive End
    else:
        prefix = "Other"

    # 提取方向
    direction = "未知"
    if "径向" in s:
        direction = "径向"
    elif "轴向" in s:
        direction = "轴向"


    return f"{prefix}_{direction}"

def split_status_label(status_str):
    """
    将 status_info 拆分为两列：
    1. run_state: '运行' 或 '停止'
    2. rpm_value: 具体数值 (float)，停止时为 0.0
    """
    if pd.isna(status_str):
        return "未知", np.nan

    s = str(status_str).strip()

    # 判断停止状态
    if "无转速" in s or "0 RPM" in s or "0RPM" in s:
        return "停止", 0.0

    # 判断运行状态并提取数值
    if "RPM" in s:
        # 使用正则提取数字 (支持小数)
        match = re.search(r"(\d+\.?\d*)", s)
        if match:
            rpm_val = float(match.group(1))
            return "运行", rpm_val
        else:
            return "运行", np.nan # 有RPM字样但没读到数

    # 其他情况
    return "未知", np.nan

# ==========================================
# 2. 读取函数 (保持不变，但在返回前可预留接口)
# ==========================================

def read_shiyu(folder_path):
    if not os.path.exists(folder_path):
        print(f"❌ 路径不存在：{folder_path}")
        return None

    last_folder = os.path.basename(folder_path)
    print(f"✅ [时域] 开始处理：{last_folder}")

    files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    if not files:
        print("   ⚠️ 未找到 .txt 文件")
        return None

    data_list = []
    for filename in files:
        file_path = os.path.join(folder_path, filename)
        try:
            match = re.search(r"(\d{14})", filename)
            if not match: continue
            timestamp_str = match.group(1)
            time_obj = pd.to_datetime(timestamp_str, format='%Y%m%d%H%M%S')

            parts = filename.split('_')
            unit_no = parts[1] if len(parts) > 1 else "Unknown"
            component_detail = parts[3] if len(parts) > 3 else "Unknown"
            status_info = parts[-1].replace('.txt', '') if parts else "Unknown"

            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read().strip()

            data_list.append({
                'timestamp': time_obj,
                'turbine_id': unit_no,
                'component_detail': component_detail,  # 原始列保留
                'status_info': status_info,  # 原始列保留
                'acceleration': content
            })
        except Exception as e:
            print(f"   读取时域文件 {filename} 出错：{e}")

    if not data_list: return None

    df = pd.DataFrame(data_list).sort_values('timestamp').reset_index(drop=True)
    print(f"   ✅ [时域] 加载完成：{len(df)} 个文件")
    return df

def read_rpm_data(folder_path):
    if not os.path.exists(folder_path):
        print(f"❌ 路径不存在：{folder_path}")
        return None

    last_folder = os.path.basename(folder_path)
    print(f"✅ [转速] 开始处理：{last_folder}")

    files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    if not files:
        print("   ⚠️ 未找到 .txt 文件")
        return None

    data_list = []
    for filename in files:
        file_path = os.path.join(folder_path, filename)
        try:
            time_match = re.search(r"(\d{14})", filename)
            if not time_match: continue
            timestamp_str = time_match.group(1)
            time_obj = pd.to_datetime(timestamp_str, format='%Y%m%d%H%M%S')

            unit_no = "Unknown"
            if '-' in filename and '_' in filename:
                first_dash = filename.find('-')
                first_under = filename.find('_')
                if first_dash < first_under:
                    unit_no = filename[first_dash+1 : first_under]

            try:
                df_content = pd.read_csv(file_path, header=None, encoding='utf-8')
                rpm_values = df_content.iloc[:, 0].tolist()
                avg_rpm = sum(rpm_values) / len(rpm_values) if rpm_values else 0
            except:
                rpm_values = []
                avg_rpm = None

            data_list.append({
                'timestamp': time_obj,
                'turbine_id': unit_no,
                'monitor_type': '转速 RPM',
                'rpm_avg': avg_rpm,
                'rpm_raw': rpm_values
            })
        except Exception as e:
            print(f"   读取转速文件 {filename} 出错：{e}")

    if not data_list: return None

    df = pd.DataFrame(data_list).sort_values('timestamp').reset_index(drop=True)
    print(f"   ✅ [转速] 加载完成：{len(df)} 个文件")
    return df


# 提取基本信息

In [3]:
# 风机编号列表
turbine_numbers = ["04", "06", "12", "14", "18", "19", "27", "28"]
base_dir = r"E:\\Grad\\data\\中广核响泉风电场"

# 【核心配置】文件夹命名模板
# 使用 {} 作为占位符，稍后用 t_num 填充
folder_template_normal = "中广核响泉风电场{}#发电机轴承更换后正常数据"
folder_template_fault = "中广核响泉风电场{}#发电机轴承故障数据"  # 根据你的描述调整此处文件名

# 用于存储所有处理后的数据 (包含正常和故障)
all_processed_dfs = []

# ==========================================
# 1. 主循环：遍历风机 -> 遍历数据类型 (正常/故障)
# ==========================================

MIN_RPM_THRESHOLD = 50.0  # 低于此转速视为非有效运行状态 (即使标签是"运行")

print(f"🚀 开始数据提取与清洗流程...")
print(f"⚙️  运行状态过滤策略：仅保留 run_state=='运行' 且 rpm > {MIN_RPM_THRESHOLD}")

# ==========================================
# 1. 主循环：遍历风机 -> 遍历数据类型
# ==========================================

for t_num in turbine_numbers:
    print(f"\n{'='*60}")
    print(f"🚜 正在处理风机：{t_num}#")
    print(f"{'='*60}")

    data_configs = [("Normal", folder_template_normal.format(t_num)),
                    ("Fault", folder_template_fault.format(t_num))]

    for category, folder_name in data_configs:
        # print(f"\n   🔍 提取 [{category}] 数据...")

        target_shiyu = os.path.join(base_dir, folder_name, "时域波形数据")
        target_rpm = os.path.join(base_dir, folder_name, "转速")

        if not os.path.exists(target_shiyu):
            # print(f"   ⚠️ 跳过：路径不存在 '{target_shiyu}'")
            continue

        try:
            # 1. 读取原始数据
            df_shiyu = read_shiyu(target_shiyu)
            
            if df_shiyu is None or df_shiyu.empty:
                continue

            original_count = len(df_shiyu)

            # 尝试读取转速
            df_rpm = None
            if os.path.exists(target_rpm):
                df_rpm = read_rpm_data(target_rpm)

            # 2. 数据清洗与重构
            # --- 拆分状态信息 ---
            try:
                # 确保 split_status_label 返回的是 [state, rpm_val] 格式
                status_split = df_shiyu['status_info'].apply(lambda x: pd.Series(split_status_label(x)))
                df_shiyu['run_state'] = status_split[0]
                df_shiyu['rpm_value'] = status_split[1].astype(float)
            except Exception as e:
                print(f"   ⚠️ 警告：拆分 status_info 失败，填充默认值。Error: {e}")
                df_shiyu['run_state'] = "Unknown"
                df_shiyu['rpm_value'] = 0.0

            # --- 标准化部件名称 ---
            df_shiyu['sensor_loc_std'] = df_shiyu['component_detail'].apply(clean_component_label)

            # --- 注入元数据 ---
            df_shiyu['turbine_id'] = t_num
            df_shiyu['data_category'] = category

            # 3. 合并转速信息
            if df_rpm is not None and not df_rpm.empty:
                df_rpm_sub = df_rpm[['timestamp', 'turbine_id', 'rpm_avg', 'rpm_raw']].copy()
                if 'turbine_id' not in df_rpm_sub.columns:
                    df_rpm_sub['turbine_id'] = t_num
                
                # 左连接，保留所有时域数据，转速缺失后面再处理
                df_shiyu = df_shiyu.merge(df_rpm_sub, on=['timestamp', 'turbine_id'], how='left')

            # 4. 【核心步骤】严格过滤：只保留“运行”且“有转速”的数据
            # print(f"    正在过滤非运行状态数据...")
            
            # A. 过滤状态列
            mask_run = df_shiyu['run_state'] == '运行'
            
            # B. 过滤转速列 (防止状态标为"运行"但实际转速为0)
            # 优先使用 merge 进来的 rpm_avg，如果没有则用解析出来的 rpm_value
            rpm_col_to_check = 'rpm_avg' if 'rpm_avg' in df_shiyu.columns else 'rpm_value'
            mask_rpm = df_shiyu[rpm_col_to_check] > MIN_RPM_THRESHOLD
            
            # 应用掩码
            df_clean = df_shiyu[mask_run & mask_rpm].copy()
            
            filtered_count = original_count - len(df_clean)
            if filtered_count > 0:
                pass # print(f"   ✂️ 已剔除 {filtered_count} 条非运行/低转速数据 (保留 {len(df_clean)} 条)")

            if df_clean.empty:
                # print(f"   ⚠️ 跳过：风机 {t_num}# [{category}] 过滤后无有效运行数据。")
                continue

            # 5. 整理列顺序与缺失值填充
            cols_order = [
                'timestamp', 'turbine_id', 'data_category', 'run_state',
                'sensor_loc_std', 'acceleration', 
                'rpm_avg', 'rpm_raw', 'rpm_value'
            ]
            final_cols = [c for c in cols_order if c in df_clean.columns]
            df_clean = df_clean[final_cols]

            # 填充转速缺失值 (虽然前面过滤了，但以防万一)
            if 'rpm_raw' in df_clean.columns:
                df_clean['rpm_raw'] = df_clean['rpm_raw'].fillna(0.0)
            if 'rpm_avg' in df_clean.columns:
                df_clean['rpm_avg'] = df_clean['rpm_avg'].fillna(df_clean['rpm_value'])

            # 6. 加入总列表
            all_processed_dfs.append(df_clean)

            # print(f"   ✅ 成功：风机 {t_num}# [{category}] -> 有效运行样本 {len(df_clean)}")

        except Exception as e:
            print(f"   ❌ 错误：处理风机 {t_num}# [{category}] 时异常 -> {e}")
            # import traceback
            # traceback.print_exc()
            continue

# ==========================================
# 2. 汇总与最终输出
# ==========================================
print(f"\n{'='*60}")
print(f"🎉 全部处理结束！(仅包含运行数据)")
print(f"{'='*60}")

if len(all_processed_dfs) > 0:
    final_df = pd.concat(all_processed_dfs, ignore_index=True)

    print(f"📦 数据合并完成:")
    print(f"   - 总有效运行行数：{len(final_df):,}")
    print(f"   - 涉及风机：{sorted(final_df['turbine_id'].unique())}")

    # 核心统计
    print(f"\n📊 运行数据类别分布:")
    print(final_df['data_category'].value_counts())

    # 交叉统计
    print(f"\n📋 各风机运行数据量详情:")
    pivot_stats = final_df.groupby(['data_category', 'turbine_id']).size().unstack(fill_value=0)
    print(pivot_stats)

    # 检查是否还有非运行数据混入 (双重保险)
    unique_states = final_df['run_state'].unique()
    if len(unique_states) > 1 or (len(unique_states) == 1 and unique_states[0] != '运行'):
        print(f"\n⚠️ 警告：最终数据中仍包含非'运行'状态：{unique_states}")
    else:
        print(f"\n✅ 验证通过：所有数据均为'运行'状态。")
        
    # 检查转速
    min_rpm = final_df['rpm_avg'].min() if 'rpm_avg' in final_df.columns else final_df['rpm_value'].min()
    print(f"✅ 验证通过：最小转速为 {min_rpm:.2f} RPM (应 > {MIN_RPM_THRESHOLD})")

    # 预览
    print(f"\n👀 数据预览 (前 5 行):")
    display_cols = ['timestamp', 'turbine_id', 'data_category', 'run_state', 'sensor_loc_std', 'rpm_avg', 'acceleration']
    display_cols = [c for c in display_cols if c in final_df.columns]
    display(final_df[display_cols].head(5))

    # 赋值全局变量
    df_final_all = final_df

else:
    print("❌ 失败：没有提取到任何有效的【运行状态】数据。请检查：\n1. status_info 解析是否正确？\n2. 文件夹中是否真的包含运行数据？\n3. 转速阈值是否设置过高？")
    df_final_all = None

🚀 开始数据提取与清洗流程...
⚙️  运行状态过滤策略：仅保留 run_state=='运行' 且 rpm > 50.0

🚜 正在处理风机：04#
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：72 个文件
✅ [转速] 开始处理：转速
   ✅ [转速] 加载完成：15 个文件
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：45 个文件
✅ [转速] 开始处理：转速
   ✅ [转速] 加载完成：8 个文件

🚜 正在处理风机：06#
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：62 个文件
✅ [转速] 开始处理：转速
   ✅ [转速] 加载完成：17 个文件
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：71 个文件
✅ [转速] 开始处理：转速
   ✅ [转速] 加载完成：23 个文件

🚜 正在处理风机：12#
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：72 个文件
✅ [转速] 开始处理：转速
   ✅ [转速] 加载完成：24 个文件
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：168 个文件
✅ [转速] 开始处理：转速
   ✅ [转速] 加载完成：56 个文件

🚜 正在处理风机：14#
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：63 个文件
✅ [转速] 开始处理：转速
   ✅ [转速] 加载完成：21 个文件
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：129 个文件
✅ [转速] 开始处理：转速
   ✅ [转速] 加载完成：23 个文件

🚜 正在处理风机：18#
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：72 个文件
✅ [转速] 开始处理：转速
   ✅ [转速] 加载完成：24 个文件
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：165 个文件
✅ [转速] 开始处理：转速
   ✅ [转速] 加载完成：55 个文件

🚜 正在处理风机：19#
✅ [时域] 开始处理：时域波形数据
   ✅ [时域] 加载完成：72 个文件
✅ [转速] 开始处理：转速
   ✅ [转速] 加载完成：

,timestamp,turbine_id,data_category,run_state,sensor_loc_std,rpm_avg,acceleration
0,2023-10-17 08:32:21,04,Normal,运行,NDE_径向,693.569204,-0.8442383\n0.7069092\n0.8236084\n0.9979248\n0...
1,2023-10-17 08:32:21,04,Normal,运行,DE_轴向,693.569204,0.9194336\n1.142822\n0.9880371\n0.4608154\n-0....
2,2023-10-17 08:32:21,04,Normal,运行,DE_径向,693.569204,-3.299194\n-2.734741\n1.837646\n2.126831\n1.42...
3,2023-10-17 16:32:24,04,Normal,运行,DE_轴向,511.069542,-1.274902\n-0.9530029\n-0.5905762\n0.000732421...
4,2023-10-17 16:32:24,04,Normal,运行,DE_径向,511.069542,-2.385132\n-1.286133\n0.05334473\n1.00061\n1.7...


# 提取时域特征

In [4]:
import numpy as np
import pandas as pd
from scipy import stats

# 检查数据框是否存在
if 'df_final_all' not in dir(
) or df_final_all is None or df_final_all.empty:
    raise ValueError("请先运行前面的合并步骤，生成 df_final_all")


# ==========================================
# 通用解析函数 (保持不变)
# ==========================================
def parse_accel_to_values(x):
    """将字符串格式的波形数据解析为 numpy 数组"""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.array([], dtype=float)
    s = str(x).strip()
    if not s:
        return np.array([], dtype=float)

    vals = []
    # 处理可能的换行符转义
    for line in s.replace("\\n", "\n").splitlines():
        line = line.strip()
        if not line:
            continue
        for part in line.split():
            try:
                vals.append(float(part))
            except ValueError:
                pass
    return np.asarray(vals, dtype=float)


def rpm_to_array(x):
    """将 rpm_raw 列转换为一维 numpy 数组"""
    from collections.abc import Sequence
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.array([], dtype=float)
    if isinstance(x, Sequence) and not isinstance(x, (str, bytes)):
        return np.asarray(list(x), dtype=float)
    try:
        return np.asarray([float(x)], dtype=float)
    except Exception:
        return np.array([], dtype=float)


# ==========================================
# 核心特征计算函数 (新增高阶指标)
# ==========================================
def calculate_time_domain_features(arr, prefix="acc"):
    """
    计算完整的时域特征集
    prefix: 列名前缀，如 'acc' 或 'rpm'
    """
    # 1. 空序列处理
    if arr.size == 0:
        return pd.Series({
            f"{prefix}_mean": np.nan,
            f"{prefix}_rms": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_var": np.nan,
            f"{prefix}_cv": np.nan,
            f"{prefix}_peak": np.nan,
            f"{prefix}_crest_factor": np.nan,
            f"{prefix}_impulse_factor": np.nan,
            f"{prefix}_margin_factor": np.nan,
            f"{prefix}_shape_factor": np.nan,
            f"{prefix}_clearance_factor": np.nan,
            f"{prefix}_q1": np.nan,
            f"{prefix}_q2": np.nan,
            f"{prefix}_q3": np.nan,
            f"{prefix}_iqr": np.nan,
            f"{prefix}_skew": np.nan,
            f"{prefix}_kurt": np.nan,
        })

    # 2. 全 0 序列处理 (避免除以零)
    if np.all(arr == 0):
        return pd.Series({
            f"{prefix}_mean": 0.0,
            f"{prefix}_rms": 0.0,
            f"{prefix}_std": 0.0,
            f"{prefix}_var": 0.0,
            f"{prefix}_cv": 0.0,
            f"{prefix}_peak": 0.0,
            f"{prefix}_crest_factor": 0.0,
            f"{prefix}_impulse_factor": 0.0,
            f"{prefix}_margin_factor": 0.0,
            f"{prefix}_shape_factor": 0.0,
            f"{prefix}_clearance_factor": 0.0,
            f"{prefix}_q1": 0.0,
            f"{prefix}_q2": 0.0,
            f"{prefix}_q3": 0.0,
            f"{prefix}_iqr": 0.0,
            f"{prefix}_skew": 0.0,
            f"{prefix}_kurt": 0.0,
        })

    # --- 基础统计量 ---
    mean_val = arr.mean()
    abs_mean = np.mean(np.abs(arr))  # 整流平均值 (用于脉冲/裕度因数)
    rms_val = np.sqrt(np.mean(arr**2))  # 均方根 (RMS)
    std_val = arr.std(ddof=1) if arr.size > 1 else 0.0
    var_val = arr.var(ddof=1) if arr.size > 1 else 0.0
    peak_val = np.max(np.abs(arr))  # 峰值

    # 变异系数 (CV) = 标准差 / 均值绝对值
    cv_val = (std_val / abs(mean_val)) if mean_val != 0 else np.nan

    # --- 无量纲指标 (故障诊断核心) ---

    # 1. 波峰因数 (Crest Factor) = 峰值 / RMS
    # 意义：>3 通常暗示有冲击，早期轴承故障敏感指标
    crest_factor = (peak_val / rms_val) if rms_val != 0 else np.nan

    # 2. 形状因数 (Shape Factor) = RMS / 整流平均值
    # 意义：反映波形形状变化
    shape_factor = (rms_val / abs_mean) if abs_mean != 0 else np.nan

    # 3. 脉冲因数 (Impulse Factor) = 峰值 / 整流平均值
    # 意义：对表面剥落等冲击性故障非常敏感
    impulse_factor = (peak_val / abs_mean) if abs_mean != 0 else np.nan

    # 4. 裕度因数 (Margin Factor) = 峰值 / (均方根幅值的某种估计，通常用 L10 或类似，这里简化为 RMS*系数 或 直接用 RMS 近似)
    # 严格定义：Peak / ( (mean(sqrt(|x|)))^2 )
    # 工程简化常用：Peak / RMS (即波峰因数) 或 Peak / (Mean_Abs)^2 * Mean_Sqrt_Abs^2
    # 这里计算标准的裕度因子定义: Peak / ( (1/N * sum(sqrt(|x|)) )^2 )
    mean_sqrt_abs = np.mean(np.sqrt(np.abs(arr)))
    margin_factor = (peak_val /
                     (mean_sqrt_abs**2)) if mean_sqrt_abs != 0 else np.nan

    # 5.  clearance factor (清除因子/裕度因子的变体) = Peak / Mean_Sqrt_Abs^2 (同上，有时定义略有不同，此处保持一致)
    # 为了区分，有时 Clearance Factor 指 Peak / RMS_LowFreq，这里我们主要关注 Margin Factor

    # --- 分布特征 ---
    q1, q2, q3 = np.quantile(arr, [0.25, 0.50, 0.75])
    iqr = q3 - q1

    # 使用 pandas 计算偏度和峭度 (Fisher 定义，正态分布峭度≈0)
    s_series = pd.Series(arr)
    skew_val = s_series.skew()
    kurt_val = s_series.kurt()

    return pd.Series({
        f"{prefix}_mean": mean_val,
        f"{prefix}_rms": rms_val,
        f"{prefix}_std": std_val,
        f"{prefix}_var": var_val,
        f"{prefix}_cv": cv_val,
        f"{prefix}_peak": peak_val,
        f"{prefix}_crest_factor": crest_factor,
        f"{prefix}_shape_factor": shape_factor,
        f"{prefix}_impulse_factor": impulse_factor,
        f"{prefix}_margin_factor": margin_factor,
        f"{prefix}_q1": q1,
        f"{prefix}_q2": q2,
        f"{prefix}_q3": q3,
        f"{prefix}_iqr": iqr,
        f"{prefix}_skew": skew_val,
        f"{prefix}_kurt": kurt_val,
    })


# ==========================================
# 执行特征提取
# ==========================================

# 1) 加速度特征
print("正在提取加速度时域特征...")
feat_df_acc = df_final_all["acceleration"].apply(
    lambda x: calculate_time_domain_features(parse_accel_to_values(x),
                                             prefix="acc"))
df_final_all = pd.concat([df_final_all, feat_df_acc], axis=1)

# 2) 转速特征
if "rpm_raw" in df_final_all.columns:
    print("正在提取转速时域特征...")
    feat_df_rpm = df_final_all["rpm_raw"].apply(
        lambda x: calculate_time_domain_features(rpm_to_array(x), prefix="rpm"
                                                 ))
    df_final_all = pd.concat([df_final_all, feat_df_rpm], axis=1)
else:
    print("⚠️ 警告：未找到 rpm_raw 列，跳过转速特征提取。")

# ==========================================
# 结果展示
# ==========================================

# 定义想要展示的列 (已更新包含新特征)
cols_show = [
    c for c in [
        'timestamp',
        'turbine_id',
        'run_state',
        'data_category',
        'sensor_loc_std',
        'acceleration',
        'rpm_raw',
        # 加速度新特征
        'acc_std',
      
        'acc_skew',
        'acc_kurt',
        # 转速新特征
        'rpm_mean',
        'rpm_std',
        'rpm_peak',
        'rpm_skew',
        'rpm_kurt',
    ] if c in df_final_all.columns
]

print(
    f"\n✅ 特征提取完成！新生成列数: {len(feat_df_acc.columns) + (len(feat_df_rpm.columns) if 'rpm_raw' in df_final_all.columns else 0)}"
)
print(f"📊 展示前 10 行关键特征 ({len(cols_show)} 列):")
df_final_all[cols_show]
display(df_final_all[cols_show])
df_final_all=df_final_all[cols_show]
# 可选：打印新特征的统计描述，检查是否有异常值 (如 inf 或 nan 比例)
# print("\n新特征统计描述:")
# print(df_final_all[[c for c in cols_show if 'acc_' in c or 'rpm_' in c]].describe())

正在提取加速度时域特征...
正在提取转速时域特征...

✅ 特征提取完成！新生成列数: 32
📊 展示前 10 行关键特征 (15 列):


,timestamp,turbine_id,run_state,data_category,sensor_loc_std,acceleration,rpm_raw,acc_std,acc_skew,acc_kurt,rpm_mean,rpm_std,rpm_peak,rpm_skew,rpm_kurt
0,2023-10-17 08:32:21,04,运行,Normal,NDE_径向,-0.8442383\n0.7069092\n0.8236084\n0.9979248\n0...,"[749.9742, 642.3865, 642.5657, 750.4116, 750.3...",1.777544,0.001519,-0.590306,693.569204,69.589401,753.5031,-0.839633,-0.252546
1,2023-10-17 08:32:21,04,运行,Normal,DE_轴向,0.9194336\n1.142822\n0.9880371\n0.4608154\n-0....,"[749.9742, 642.3865, 642.5657, 750.4116, 750.3...",1.375636,-0.012914,-0.066564,693.569204,69.589401,753.5031,-0.839633,-0.252546
2,2023-10-17 08:32:21,04,运行,Normal,DE_径向,-3.299194\n-2.734741\n1.837646\n2.126831\n1.42...,"[749.9742, 642.3865, 642.5657, 750.4116, 750.3...",1.262370,-0.118427,-0.145496,693.569204,69.589401,753.5031,-0.839633,-0.252546
3,2023-10-17 16:32:24,04,运行,Normal,DE_轴向,-1.274902\n-0.9530029\n-0.5905762\n0.000732421...,"[416.4268, 352.1215, 381.6927, 572.6375, 352.5...",1.274668,-0.014762,-0.146160,511.069542,96.705320,761.6105,0.223517,-0.219100
4,2023-10-17 16:32:24,04,运行,Normal,DE_径向,-2.385132\n-1.286133\n0.05334473\n1.00061\n1.7...,"[416.4268, 352.1215, 381.6927, 572.6375, 352.5...",1.328846,-0.105722,-0.199303,511.069542,96.705320,761.6105,0.223517,-0.219100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1314,2023-10-19 13:12:45,28,运行,Fault,NDE_径向,-1.449951\n-0.510498\n-0.927002\n-0.6717529\n-...,"[1081.171, 1081.366, 1082.817, 1083.707, 1083....",6.931933,-0.288248,3.715290,1087.413762,8.692032,1108.2920,0.343361,-0.398928
1315,2023-10-19 13:12:45,28,运行,Fault,DE_径向,1.024292\n2.715576\n3.046143\n0.9029541\n0.565...,"[1081.171, 1081.366, 1082.817, 1083.707, 1083....",2.732276,0.166496,0.424253,1087.413762,8.692032,1108.2920,0.343361,-0.398928
1316,2023-10-19 15:12:45,28,运行,Fault,NDE_径向,0.4580078\n0.935791\n-7.524048\n-4.655029\n2.6...,"[1101.401, 1101.991, 1102.728, 1103.161, 1102....",7.446057,-0.143282,2.825657,1101.144840,5.465575,1113.0900,-0.165900,-0.600014
1317,2023-10-19 15:12:45,28,运行,Fault,DE_径向,-1.483887\n0.6230469\n0.7521973\n-0.18396\n2.6...,"[1101.401, 1101.991, 1102.728, 1103.161, 1102....",2.828395,-0.049565,0.265328,1101.144840,5.465575,1113.0900,-0.165900,-0.600014


# 保存时域特征提取结果

In [ ]:
# import os
# import pandas as pd

# # 1. 定义路径
# save_dir = r"E:\\Grad\\data\\关键中转数据"
# file_path = os.path.join(save_dir, "时域特征提取后.pkl")  # 后缀改为 .pkl

# # 2. 确保文件夹存在
# os.makedirs(save_dir, exist_ok=True)

# # 3. 保存为 Pickle (protocol=5 是 Python 3.8+ 的高效协议，适合大数据)
# df_final_all.to_pickle(file_path, protocol=5)

# print(f"✅ [Pickle] 成功保存至：{file_path}")
# print(f"📦 文件大小：{os.path.getsize(file_path) / (1024*1024):.2f} MB")

import pandas as pd

file_path = r"E:\\Grad\\data\\关键中转数据\\时域特征提取后.pkl"
df_final_all = pd.read_pickle(file_path)

display(df_final_all.head())

✅ [Pickle] 成功保存至：E:\\Grad\\data\\关键中转数据\时域特征提取后.pkl
📦 文件大小：1261.88 MB


,timestamp,turbine_id,run_state,data_category,sensor_loc_std,acceleration,rpm_raw,acc_std,acc_skew,acc_kurt,rpm_mean,rpm_std,rpm_peak,rpm_skew,rpm_kurt
0,2023-10-17 08:32:21,04,运行,Normal,NDE_径向,-0.8442383\n0.7069092\n0.8236084\n0.9979248\n0...,"[749.9742, 642.3865, 642.5657, 750.4116, 750.3...",1.777544,0.001519,-0.590306,693.569204,69.589401,753.5031,-0.839633,-0.252546
1,2023-10-17 08:32:21,04,运行,Normal,DE_轴向,0.9194336\n1.142822\n0.9880371\n0.4608154\n-0....,"[749.9742, 642.3865, 642.5657, 750.4116, 750.3...",1.375636,-0.012914,-0.066564,693.569204,69.589401,753.5031,-0.839633,-0.252546
2,2023-10-17 08:32:21,04,运行,Normal,DE_径向,-3.299194\n-2.734741\n1.837646\n2.126831\n1.42...,"[749.9742, 642.3865, 642.5657, 750.4116, 750.3...",1.262370,-0.118427,-0.145496,693.569204,69.589401,753.5031,-0.839633,-0.252546
3,2023-10-17 16:32:24,04,运行,Normal,DE_轴向,-1.274902\n-0.9530029\n-0.5905762\n0.000732421...,"[416.4268, 352.1215, 381.6927, 572.6375, 352.5...",1.274668,-0.014762,-0.146160,511.069542,96.705320,761.6105,0.223517,-0.219100
4,2023-10-17 16:32:24,04,运行,Normal,DE_径向,-2.385132\n-1.286133\n0.05334473\n1.00061\n1.7...,"[416.4268, 352.1215, 381.6927, 572.6375, 352.5...",1.328846,-0.105722,-0.199303,511.069542,96.705320,761.6105,0.223517,-0.219100


# 提取频域特征

In [6]:
import numpy as np
import pandas as pd
from scipy import signal
from scipy.fft import rfft, rfftfreq
import warnings
import gc # 用于垃圾回收，防止内存溢出

warnings.filterwarnings('ignore')

# ==========================================
# 1. 配置参数
# ==========================================
FS = 25600              # 采样率
# 不需要设置频率上限，因为我们要保留所有频点以匹配格式逻辑 (虽然点数会是 N/2+1)
DECIMAL_PRECISION = 6   # 保留 6 位小数，保证精度，同时控制字符串长度

print(f"⚙️ 开始计算全频段 FFT 并格式化...")
print(f"⚠️ 警告：生成的字符串将非常长 (每行约 51k 个数值)，请注意内存使用情况。")

# ==========================================
# 2. 辅助函数：解析原始波形
# ==========================================
def parse_accel_to_values(x):
    """将字符串格式的波形数据解析为 numpy 数组"""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.array([], dtype=float)
    s = str(x).strip()
    if not s:
        return np.array([], dtype=float)
    
    # 优化解析速度：直接使用 split 处理所有空白字符 (包括 \n 和空格)
    # 假设数据之间由空白字符分隔
    try:
        # 替换可能的转义换行符
        s_clean = s.replace("\\n", " ")
        vals = np.fromstring(s_clean, sep=' ')
        if len(vals) == 0:
            # 如果 fromstring 失败，尝试手动解析 (兼容旧格式)
            vals = []
            for part in s_clean.split():
                try:
                    vals.append(float(part))
                except ValueError:
                    pass
            return np.asarray(vals, dtype=float)
        return vals
    except Exception:
        return np.array([], dtype=float)

# ==========================================
# 3. 核心函数：计算 FFT 并输出空格分隔字符串
# ==========================================
def compute_fft_string_matching_format(waveform_str, fs, precision=6):
    """
    输入：原始波形字符串 (102400 点)
    输出：频域幅值字符串 (51201 点)，格式为 "val1 val2 val3 ..." (空格分隔)
    """
    # 1. 解析波形
    sig = parse_accel_to_values(waveform_str)
    n = len(sig)
    
    if n < 1024: 
        return ""
    
    # 2. 预处理：去均值 + 加汉宁窗
    sig = sig - np.mean(sig)
    window = signal.windows.hann(n)
    sig_win = sig * window
    
    # 3. 执行 FFT (rfft 返回 N/2 + 1 个点)
    fft_vals = rfft(sig_win)
    # freqs = rfftfreq(n, 1/fs) # 如果需要频率轴可以取消注释，但这里只存幅值
    
    # 4. 计算幅值谱 (修正窗函数增益)
    # 物理幅值 = 2 * |FFT| / sum(window)
    amps = 2.0 * np.abs(fft_vals) / np.sum(window)
    
    # 5. 格式化为字符串 (空格分隔，模仿原始 acceleration 列)
    # 使用 numpy 的 array2string 或者列表推导式
    # 为了严格控制小数位数，使用列表推导式
    # 注意：这步最耗时且最占内存
    formatted_vals = [f"{v:.{precision}f}" for v in amps]
    
    return " ".join(formatted_vals)

# ==========================================
# 4. 执行转换
# ==========================================

if 'df_final_all' not in locals() or df_final_all is None:
    raise ValueError("df_final_all 不存在。")

if 'acceleration' not in df_final_all.columns:
    raise ValueError("未找到 'acceleration' 列。")

# 检查第一行数据长度以确认预期
sample_len = len(parse_accel_to_values(df_final_all['acceleration'].iloc[0]))
expected_fft_len = (sample_len // 2) + 1
print(f"检测到原始信号长度：{sample_len} 点")
print(f"预期 FFT 结果长度：{expected_fft_len} 点")

# 执行应用
# 提示：如果数据量大，建议使用 tqdm 显示进度，这里为了简洁直接运行
print("正在逐行计算 FFT 并格式化 (可能需要较长时间)...")

# 为了节省内存，分块处理或者直接 apply (视内存而定)
# 这里直接 apply，如果内存报错，建议分批处理
df_final_all['acceleration_fft'] = df_final_all['acceleration'].apply(
    lambda x: compute_fft_string_matching_format(x, FS, DECIMAL_PRECISION)
)

# 强制垃圾回收
gc.collect()

print(f"✅ 完成！已新增列 'acceleration_fft'。")

# ==========================================
# 5. 验证与展示
# ==========================================

def count_points_in_string(s):
    if not s or not isinstance(s, str):
        return 0
    return len(s.split())

# 验证前 3 行
print("\n📊 格式验证:")
for i in range(min(3, len(df_final_all))):
    orig_str = df_final_all['acceleration'].iloc[i]
    fft_str = df_final_all['acceleration_fft'].iloc[i]
    
    orig_count = count_points_in_string(orig_str)
    fft_count = count_points_in_string(fft_str)
    
    print(f"Row {i}:")
    print(f"  Original Points: {orig_count}")
    print(f"  FFT Points     : {fft_count} (Expected: {orig_count//2 + 1})")
    print(f"  Original Preview: {orig_str[:50]}...")
    print(f"  FFT Preview     : {fft_str[:50]}...")
    print("-" * 30)

# 统计字符串长度分布
df_final_all['fft_str_len'] = df_final_all['acceleration_fft'].apply(lambda x: len(x) if isinstance(x, str) else 0)
print(f"\n📈 统计信息:")
print(f"平均 FFT 字符串长度：{df_final_all['fft_str_len'].mean():.0f} 字符")
print(f"最大 FFT 字符串长度：{df_final_all['fft_str_len'].max():.0f} 字符")
print(f"估算总内存占用 (仅该列): {df_final_all['fft_str_len'].sum() / 1024 / 1024:.2f} MB")

# 清理临时列
df_final_all.drop(columns=['fft_str_len'], inplace=True)

print("\n💡 提示：'acceleration_fft' 列现在包含了空格分隔的频域幅值数据。")
print("   解析方法：np.fromstring(row['acceleration_fft'], sep=' ')")

⚙️ 开始计算全频段 FFT 并格式化...
⚠️ 警告：生成的字符串将非常长 (每行约 51k 个数值)，请注意内存使用情况。
检测到原始信号长度：102400 点
预期 FFT 结果长度：51201 点
正在逐行计算 FFT 并格式化 (可能需要较长时间)...
✅ 完成！已新增列 'acceleration_fft'。

📊 格式验证:
Row 0:
  Original Points: 102400
  FFT Points     : 51201 (Expected: 51201)
  Original Preview: -0.8442383
0.7069092
0.8236084
0.9979248
0.657959
...
  FFT Preview     : 0.010972 0.019408 0.014644 0.006702 0.005603 0.001...
------------------------------
Row 1:
  Original Points: 102400
  FFT Points     : 51201 (Expected: 51201)
  Original Preview: 0.9194336
1.142822
0.9880371
0.4608154
-0.1318359
...
  FFT Preview     : 0.012066 0.023102 0.062567 0.046852 0.002177 0.024...
------------------------------
Row 2:
  Original Points: 102400
  FFT Points     : 51201 (Expected: 51201)
  Original Preview: -3.299194
-2.734741
1.837646
2.126831
1.421387
1.5...
  FFT Preview     : 0.003128 0.010294 0.008334 0.003554 0.002398 0.004...
------------------------------

📈 统计信息:
平均 FFT 字符串长度：460808 字符
最大 FFT 字符串长度：460808 字符
估算总内存占用

In [7]:
import numpy as np
import pandas as pd
from scipy import stats
import warnings
import gc
import os

# 忽略警告
warnings.filterwarnings('ignore')


# ==========================================
# 1. 配置参数
# ==========================================
FS = 25600  # 采样率
# 频段定义 (Hz)
FREQ_LOW_LIMIT = 10.0   
FREQ_MID_LIMIT = 200.0  

print("⚙️ 开始提取【转速无关】的核心频域形态特征 (含熵与重心)...")

# ==========================================
# 2. 辅助函数：解析 FFT 字符串
# ==========================================
def parse_fft_string(s):
    """高效解析空格分隔的字符串为 numpy 数组"""
    if not isinstance(s, str) or len(s) < 5:
        return np.array([])
    try:
        return np.fromstring(s, sep=' ')
    except Exception:
        return np.array([])

# ==========================================
# 3. 核心特征提取函数 (已包含熵和重心)
# ==========================================
def extract_invariant_features(fft_str):
    """
    输入：FFT 幅值字符串
    输出：字典，包含修正后的 5 个核心特征
    """
    amps = parse_fft_string(fft_str)
    n = len(amps)
    
    default_nan = {
        'spectral_kurtosis_acc_fft': np.nan,
        'crest_factor_acc_fft': np.nan,
        'energy_high_ratio_acc_fft': np.nan,
        'spectral_entropy_acc_fft': np.nan,
        'spectral_centroid_acc_fft': np.nan,
        '_rms_check': np.nan 
    }
    
    if n < 10: 
        return default_nan
    
    # 生成频率轴
    freqs = np.linspace(0, FS/2, n)
    
    # 预处理：去除直流分量
    amps_clean = amps.copy()
    amps_clean[0] = 0.0
    
    if np.all(amps_clean == 0):
        return default_nan

    # --- 核心基础：功率谱 (Energy Spectrum) ---
    # 所有频域统计特征应基于能量 (幅值的平方) 计算，以符合物理意义
    power_spectrum = amps_clean ** 2
    total_power = np.sum(power_spectrum)
    
    # 防止除以零
    if total_power <= 1e-9:
        return default_nan

    # --- A. 基础统计量 (用于峰值因子) ---
    # RMS 基于能量计算
    rms_total = np.sqrt(np.mean(power_spectrum)) 
    peak_amp = np.max(amps_clean) # 峰值通常看幅值
    
    # 1. 峰值因子 (Crest Factor) = Peak / RMS
    crest_factor = (peak_amp / rms_total) if rms_total > 1e-9 else 0.0
    
    # --- B. 能量占比 (Energy Ratios) ---
    mask_high = freqs > FREQ_MID_LIMIT
    e_high = np.sum(power_spectrum[mask_high]) / total_power
    
    # --- C. 频谱峭度 (Spectral Kurtosis) ---
    # 对幅值谱计算峭度 (工程常用) 或 对功率谱计算 (更严格)
    # 这里保持原逻辑：对幅值谱计算，因为故障尖峰在幅值谱上也很明显
    kurt_val = stats.kurtosis(amps_clean, fisher=False)
    if np.isnan(kurt_val): kurt_val = 0.0

    # --- D. 🆕 修正后的频谱熵 (基于能量概率分布) ---
    p_norm = power_spectrum / total_power
    # 使用 log2 (单位 bits)，若需 nats 可改回 np.log
    spectral_entropy = -np.sum(p_norm * np.log2(p_norm + 1e-9))
    
    # --- E. 🆕 修正后的频谱重心 (基于能量的加权平均频率) ---
    # 公式：Sum(f * P(f)) / Sum(P(f))
    spectral_centroid = np.sum(freqs * power_spectrum) / total_power

    return {
        'spectral_kurtosis_acc_fft': kurt_val,
        'crest_factor_acc_fft': crest_factor,
        'energy_high_ratio_acc_fft': e_high,
        'spectral_entropy_acc_fft': spectral_entropy,
        'spectral_centroid_acc_fft': spectral_centroid,
        '_rms_check': rms_total 
    }

# ==========================================
# 4. 执行提取
# ==========================================
if 'acceleration_fft' not in df_final_all.columns:
    raise ValueError("错误：未找到 'acceleration_fft' 列。请先运行 FFT 转换代码。")

print(f"🔄 正在逐行计算 5+2 个核心形态特征 (这可能需要几分钟)...")

# 应用函数
features_series = df_final_all['acceleration_fft'].apply(extract_invariant_features)

# 转换为 DataFrame
df_features = pd.DataFrame(features_series.tolist(), index=df_final_all.index)

# ==========================================
# 5. 数据清洗与后处理
# ==========================================

# A. 过滤无效行
valid_mask = (df_features['_rms_check'] > 1e-9) & (~df_features['_rms_check'].isna())
invalid_count = (~valid_mask).sum()

if invalid_count > 0:
    print(f"🗑️ 检测到 {invalid_count} 行无效数据，将设为 NaN。")
    cols_to_nan = [
        'spectral_kurtosis_acc_fft', 'crest_factor_acc_fft', 
        'energy_high_ratio_acc_fft', 'spectral_entropy_acc_fft', 
        'spectral_centroid_acc_fft'
    ]
    df_features.loc[~valid_mask, cols_to_nan] = np.nan

# 删除临时检查列
df_features.drop(columns=['_rms_check'], inplace=True)

# B. 对数变换 (针对峭度)
col_kurt = 'spectral_kurtosis_acc_fft'
print(f"\n🔄 正在对 '{col_kurt}' 进行对数变换 (log1p)...")
if col_kurt in df_features.columns:
    orig_max = df_features[col_kurt].max()
    df_features[col_kurt] = np.log1p(df_features[col_kurt])
    print(f"   变换前最大值：{orig_max:.2f} -> 变换后：{df_features[col_kurt].max():.4f}")

# (可选) 如果您发现熵的分布也非常偏斜，也可以在这里对 entropy 做 log 变换
# df_features['spectral_entropy_acc_fft'] = np.log1p(df_features['spectral_entropy_acc_fft'])

# ==========================================
# 6. 合并到主表
# ==========================================

final_cols_to_add = [
    'spectral_kurtosis_acc_fft', 
    'crest_factor_acc_fft',
    'energy_high_ratio_acc_fft',
    'spectral_entropy_acc_fft',   # 🆕 新增列
    'spectral_centroid_acc_fft'   # 🆕 新增列
]

# 检查重名并覆盖
existing_overlap = [c for c in final_cols_to_add if c in df_final_all.columns]
if existing_overlap:
    print(f"⚠️ 发现已存在的列 {existing_overlap}，正在覆盖...")
    df_final_all.drop(columns=existing_overlap, inplace=True)

# 合并
df_final_all = df_final_all.join(df_features[final_cols_to_add])

gc.collect()

print("\n✅ 所有核心形态特征 (含熵与重心) 提取完成！")

# ==========================================
# 7. 最终验证与统计
# ==========================================
print("\n📈 特征统计摘要:")
stats_display = df_final_all[final_cols_to_add].describe().loc[['count', 'mean', 'std', 'min', 'max']]
print(stats_display)

# 能量比校验
valid_rows_for_check = df_final_all[final_cols_to_add].dropna()
if len(valid_rows_for_check) > 0:
    # 注意：这里只校验了高频占比，若要校验总和需重新计算低中频，此处略过复杂校验
    print(f"\n💡 数据概览:")
    print(f"   有效样本数：{len(valid_rows_for_check)}")
    print(f"   频谱重心均值：{valid_rows_for_check['spectral_centroid_acc_fft'].mean():.2f} Hz")
    print(f"   频谱熵均值：{valid_rows_for_check['spectral_entropy_acc_fft'].mean():.4f}")

print("\n📋 最终 DataFrame 包含的关键频域特征列:")
print([c for c in df_final_all.columns if any(keyword in c for keyword in ['kurtosis', 'crest', 'energy_', 'entropy', 'centroid'])])

print("\n💡 下一步建议:")
print("   1. 'spectral_entropy_acc_fft' (混乱度) 和 'spectral_centroid_acc_fft' (能量中心) 已就绪。")
print("   2. 故障恶化时，通常观察到：熵增加 (更乱)，重心右移 (高频增多)。")
print("   3. 记得在训练前 dropna() 处理掉无效行。")
print("   4. 保存更新后的数据：df_final_all.to_pickle('...')")

⚙️ 开始提取【转速无关】的核心频域形态特征 (含熵与重心)...
🔄 正在逐行计算 5+2 个核心形态特征 (这可能需要几分钟)...

🔄 正在对 'spectral_kurtosis_acc_fft' 进行对数变换 (log1p)...
   变换前最大值：14889.78 -> 变换后：9.6085

✅ 所有核心形态特征 (含熵与重心) 提取完成！

📈 特征统计摘要:
       spectral_kurtosis_acc_fft  crest_factor_acc_fft  \
count                1319.000000           1319.000000   
mean                    5.119044             35.490736   
std                     1.562625             20.390900   
min                     1.792731              5.365300   
max                     9.608498            144.002734   

       energy_high_ratio_acc_fft  spectral_entropy_acc_fft  \
count                1319.000000               1319.000000   
mean                    0.978956                 11.418757   
std                     0.048521                  1.691898   
min                     0.060567                  3.428243   
max                     0.999931                 14.179387   

       spectral_centroid_acc_fft  
count                1319.000000  
mean            

# 保存频域特征提取结果

In [ ]:
# import os
# import pandas as pd

# # 1. 定义路径
# save_dir = r"E:\\Grad\\data\\关键中转数据"
# file_path = os.path.join(save_dir, "频域特征提取后.pkl")  # 后缀改为 .pkl

# # 2. 确保文件夹存在
# os.makedirs(save_dir, exist_ok=True)

# # 3. 保存为 Pickle (protocol=5 是 Python 3.8+ 的高效协议，适合大数据)
# df_final_all.to_pickle(file_path, protocol=5)

# print(f"✅ [Pickle] 成功保存至：{file_path}")
# print(f"📦 文件大小：{os.path.getsize(file_path) / (1024*1024):.2f} MB")

import pandas as pd

file_path = r"E:\\Grad\\data\\关键中转数据\\频域特征提取后.pkl"
df_final_all = pd.read_pickle(file_path)

display(df_final_all.head())

✅ [Pickle] 成功保存至：E:\\Grad\\data\\关键中转数据\频域特征提取后.pkl
📦 文件大小：1841.59 MB


,timestamp,turbine_id,run_state,data_category,sensor_loc_std,acceleration,rpm_raw,acc_std,acc_skew,acc_kurt,...,rpm_std,rpm_peak,rpm_skew,rpm_kurt,acceleration_fft,spectral_kurtosis_acc_fft,crest_factor_acc_fft,energy_high_ratio_acc_fft,spectral_entropy_acc_fft,spectral_centroid_acc_fft
0,2023-10-17 08:32:21,04,运行,Normal,NDE_径向,-0.8442383\n0.7069092\n0.8236084\n0.9979248\n0...,"[749.9742, 642.3865, 642.5657, 750.4116, 750.3...",1.777544,0.001519,-0.590306,...,69.589401,753.5031,-0.839633,-0.252546,0.010972 0.019408 0.014644 0.006702 0.005603 0...,8.594857,115.166980,0.995175,6.616584,3220.209433
1,2023-10-17 08:32:21,04,运行,Normal,DE_轴向,0.9194336\n1.142822\n0.9880371\n0.4608154\n-0....,"[749.9742, 642.3865, 642.5657, 750.4116, 750.3...",1.375636,-0.012914,-0.066564,...,69.589401,753.5031,-0.839633,-0.252546,0.012066 0.023102 0.062567 0.046852 0.002177 0...,6.700683,54.760497,0.995643,7.878222,2915.103583
2,2023-10-17 08:32:21,04,运行,Normal,DE_径向,-3.299194\n-2.734741\n1.837646\n2.126831\n1.42...,"[749.9742, 642.3865, 642.5657, 750.4116, 750.3...",1.262370,-0.118427,-0.145496,...,69.589401,753.5031,-0.839633,-0.252546,0.003128 0.010294 0.008334 0.003554 0.002398 0...,7.702690,82.435446,0.992233,8.268271,4033.350576
3,2023-10-17 16:32:24,04,运行,Normal,DE_轴向,-1.274902\n-0.9530029\n-0.5905762\n0.000732421...,"[416.4268, 352.1215, 381.6927, 572.6375, 352.5...",1.274668,-0.014762,-0.146160,...,96.705320,761.6105,0.223517,-0.219100,0.002742 0.010710 0.027205 0.019433 0.007800 0...,7.729382,74.168031,0.997365,6.795559,2975.827482
4,2023-10-17 16:32:24,04,运行,Normal,DE_径向,-2.385132\n-1.286133\n0.05334473\n1.00061\n1.7...,"[416.4268, 352.1215, 381.6927, 572.6375, 352.5...",1.328846,-0.105722,-0.199303,...,96.705320,761.6105,0.223517,-0.219100,0.006931 0.017067 0.020207 0.007631 0.002921 0...,7.712872,84.414884,0.994741,7.820619,3933.128104


# 包络分析

In [9]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import hilbert, butter, filtfilt
import warnings
import time

warnings.filterwarnings('ignore')

# ==========================================
# 1. 辅助函数定义
# ==========================================

def parse_and_clean_signal(s, threshold=500):
    """解析字符串信号并去除异常尖峰"""
    if not isinstance(s, str) or len(s) < 5:
        return np.array([])
    try:
        # 兼容多种分隔符
        s_clean = s.replace('\\n', ' ').replace('\n', ' ')
        sig = np.fromstring(s_clean, sep=' ')
        
        # 处理可能的空解析
        if len(sig) == 0:
            # 尝试逗号分隔
            sig = np.fromstring(s_clean.replace(',', ' '), sep=' ')
            
        sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
        # 去除超过阈值的异常值 (设为0)
        sig[np.abs(sig) > threshold] = 0.0 
        return sig
    except Exception:
        return np.array([])

def extract_advanced_features(sig, fs, f_min=200, f_max=10000, max_level=4):
    """
    核心函数：计算谱峭度最佳频带 + 包络特征
    (已移除小波包熵)
    返回: dict {best_low, best_high, max_sk, env_kurt, env_crest}
    """
    n = len(sig)
    default_result = {
        'best_low': np.nan, 'best_high': np.nan, 
        'max_sk': np.nan, 'env_kurt': np.nan, 
        'env_crest': np.nan
    }
    
    if n < 100:
        return default_result
    
    # 去均值
    sig = sig - np.mean(sig)
    if np.std(sig) < 1e-9:
        return default_result
    
    nyq = fs / 2
    best_sk = -np.inf
    best_band = (np.nan, np.nan)
    best_filtered_sig = None
    
    # --- 步骤 1: 寻找谱峭度最大的频带 (频带搜索) ---
    epsilon = 1e-6
    
    # 优化：如果信号很长，可以先降采样加速搜索，但这里保持原样以保证精度
    for level in range(1, max_level + 1):
        num_bands = 2**level
        bw = nyq / num_bands
        
        for i in range(num_bands):
            f_low_raw = i * bw
            f_high_raw = (i + 1) * bw
            
            # 只关注感兴趣的故障频带
            if f_high_raw <= f_min or f_low_raw >= f_max:
                continue
            
            f_low = max(f_low_raw, f_min)
            f_high = min(f_high_raw, f_max)
            
            if f_low >= f_high:
                continue
            
            low_norm = max(epsilon, f_low / nyq)
            high_norm = min(1.0 - epsilon, f_high / nyq)
            
            if low_norm >= high_norm:
                continue
            
            try:
                # 4阶巴特沃斯带通滤波
                b, a = butter(4, [low_norm, high_norm], btype='bandpass')
                y = filtfilt(b, a, sig)
                
                if np.std(y) < 1e-9 or np.any(np.isnan(y)):
                    continue
                
                # 计算峭度 (fisher=False 返回 Pearson 定义，即 E[x^4]/std^4)
                k = stats.kurtosis(y, fisher=False)
                
                if k > best_sk:
                    best_sk = k
                    best_band = (f_low, f_high)
                    best_filtered_sig = y
                    
            except Exception:
                continue
    
    if np.isnan(best_band[0]) or best_filtered_sig is None:
        return default_result
    
    # --- 步骤 2: 基于最佳频带计算包络特征 ---
    # 希尔伯特变换提取包络
    envelope = np.abs(hilbert(best_filtered_sig))
    
    # 1. 包络峭度 (反映冲击的尖锐程度)
    env_kurt = stats.kurtosis(envelope, fisher=False)
    
    # 2. 包络峰值因子 (Crest Factor = Peak / RMS)
    env_rms = np.sqrt(np.mean(envelope**2))
    env_peak = np.max(envelope)
    env_crest = env_peak / (env_rms + 1e-9)
    
    # --- 步骤 3: (已移除小波包熵) ---
    
    return {
        'best_low': best_band[0],
        'best_high': best_band[1],
        'max_sk': best_sk,
        'env_kurt': env_kurt,
        'env_crest': env_crest
    }

# ==========================================
# 2. 主执行流程
# ==========================================

print("⚙️ 开始执行高级特征提取 (谱峭度 + 包络分析)...")
print("   [已移除小波包模块，运行速度将提升]")

FS = 25600  # 采样率
TARGET_COL = 'acceleration'

# 检查数据
if 'df_final_all' not in globals():
    raise ValueError("❌ 错误：未找到 DataFrame 'df_final_all'")
if TARGET_COL not in df_final_all.columns:
    raise ValueError(f"❌ 错误：未找到列 '{TARGET_COL}'")

# 初始化结果列表
res_low, res_high, res_sk = [], [], []
res_env_k, res_env_c = [], []

start_time = time.time()
total_rows = len(df_final_all)
print(f"📊 正在处理 {total_rows} 行数据...")

debug_printed = False

for idx, row in df_final_all.iterrows():
    sig = parse_and_clean_signal(row[TARGET_COL])
    
    if len(sig) < 100:
        # 填充 NaN
        res_low.append(np.nan); res_high.append(np.nan); res_sk.append(np.nan)
        res_env_k.append(np.nan); res_env_c.append(np.nan)
        continue
    
    # 提取特征
    feats = extract_advanced_features(sig, FS)
    
    res_low.append(feats['best_low'])
    res_high.append(feats['best_high'])
    res_sk.append(feats['max_sk'])
    res_env_k.append(feats['env_kurt'])
    res_env_c.append(feats['env_crest'])
    
    # 打印首个成功样本
    if not debug_printed and not np.isnan(feats['best_low']):
        print(f"✅ 首个样本成功! 频带:{feats['best_low']:.1f}-{feats['best_high']:.1f}Hz, "
              f"SK:{feats['max_sk']:.2f}, 包络峭度:{feats['env_kurt']:.2f}, "
              f"包络峰值因子:{feats['env_crest']:.2f}")
        debug_printed = True
    
    if (idx + 1) % 500 == 0:
        print(f"   ... 进度：{idx + 1}/{total_rows}")

elapsed = time.time() - start_time
print(f"\n✅ 全部完成！总耗时：{elapsed:.2f} 秒")

# ==========================================
# 3. 结果汇总与写入 DataFrame
# ==========================================

# 将结果写入 DataFrame

df_final_all['max_kurtosis'] = res_sk        # 谱峭度 (核心)
df_final_all['env_kurtosis'] = res_env_k     # 包络峭度 (核心)
df_final_all['env_crest_factor'] = res_env_c # 包络峰值因子 (核心)

# 创建有效数据子集用于展示
df_valid = df_final_all.dropna(subset=['max_kurtosis']).copy()

if not df_valid.empty:
    print("\n" + "="*70)
    print("📈 特征提取结果预览 (前 10 行)")
    print("="*70)
    display_cols = ['max_kurtosis', 'env_kurtosis', 'env_crest_factor']
    if 'turbine_id' in df_valid.columns:
        display_cols.insert(0, 'turbine_id')
    if 'data_category' in df_valid.columns:
        display_cols.insert(len(display_cols), 'data_category')
    
    # 格式化显示
    pd.set_option('display.precision', 3)
    print(df_valid[display_cols].head(10).to_string())
    
    print("\n" + "="*70)
    print("📊 特征统计摘要")
    print("="*70)
    stats_df = df_valid[['max_kurtosis', 'env_kurtosis', 'env_crest_factor']].describe()
    print(stats_df)
    
    print("\n💡 关键发现:")
    print(f"   - 平均谱峭度 (SK): {df_valid['max_kurtosis'].mean():.2f}")
    print(f"   - 平均包络峭度: {df_valid['env_kurtosis'].mean():.2f}")
    print(f"   - 平均包络峰值因子: {df_valid['env_crest_factor'].mean():.2f}")
    print(f"   (注：已移除小波包熵，计算效率提升)")
    
else:
    print("\n❌ 未生成任何有效特征。请检查输入信号是否为空或全零。")

# 最终可用的特征列列表
final_feature_cols = ['max_kurtosis', 'env_kurtosis', 'env_crest_factor']
print(f"\n🎯 准备好用于模型训练的特征列: {final_feature_cols}")

⚙️ 开始执行高级特征提取 (谱峭度 + 包络分析)...
   [已移除小波包模块，运行速度将提升]
📊 正在处理 1319 行数据...
✅ 首个样本成功! 频带:4000.0-4800.0Hz, SK:4.79, 包络峭度:6.83, 包络峰值因子:4.43
   ... 进度：500/1319
   ... 进度：1000/1319

✅ 全部完成！总耗时：283.94 秒

📈 特征提取结果预览 (前 10 行)
  turbine_id  max_kurtosis  env_kurtosis  env_crest_factor data_category
0         04         4.794         6.826             4.428        Normal
1         04         5.560        11.518             5.773        Normal
2         04        19.103        99.626            14.205        Normal
3         04         4.345         8.108             6.156        Normal
4         04         5.344        11.277             7.809        Normal
5         04         6.163        11.568             6.372        Normal
6         04         4.482         7.443             6.261        Normal
7         04         4.924         8.912             6.035        Normal
8         04         4.273         6.087             4.459        Normal
9         04         4.243         5.980             5.1

# 保存包络分析特征提取结果

In [29]:
# import os
# import pandas as pd

# # 1. 定义路径
# save_dir = r"E:\\Grad\\data\\关键中转数据"
# file_path = os.path.join(save_dir, "包络分析特征提取后.pkl")  # 后缀改为 .pkl

# # 2. 确保文件夹存在
# os.makedirs(save_dir, exist_ok=True)

# # 3. 保存为 Pickle (protocol=5 是 Python 3.8+ 的高效协议，适合大数据)
# df_final_all.to_pickle(file_path, protocol=5)

# print(f"✅ [Pickle] 成功保存至：{file_path}")
# print(f"📦 文件大小：{os.path.getsize(file_path) / (1024*1024):.2f} MB")

import pandas as pd

file_path = r"E:\\Grad\\data\\关键中转数据\\包络分析特征提取后.pkl"
df_final_all = pd.read_pickle(file_path)

display(df_final_all.head())

,timestamp,turbine_id,run_state,data_category,sensor_loc_std,acceleration,rpm_raw,acc_std,acc_skew,acc_kurt,...,rpm_kurt,acceleration_fft,spectral_kurtosis_acc_fft,crest_factor_acc_fft,energy_high_ratio_acc_fft,spectral_entropy_acc_fft,spectral_centroid_acc_fft,max_kurtosis,env_kurtosis,env_crest_factor
0,2023-10-17 08:32:21,04,运行,Normal,NDE_径向,-0.8442383\n0.7069092\n0.8236084\n0.9979248\n0...,"[749.9742, 642.3865, 642.5657, 750.4116, 750.3...",1.778,0.002,-0.590,...,-0.253,0.010972 0.019408 0.014644 0.006702 0.005603 0...,8.595,115.167,0.995,6.617,3220.209,4.794,6.826,4.428
1,2023-10-17 08:32:21,04,运行,Normal,DE_轴向,0.9194336\n1.142822\n0.9880371\n0.4608154\n-0....,"[749.9742, 642.3865, 642.5657, 750.4116, 750.3...",1.376,-0.013,-0.067,...,-0.253,0.012066 0.023102 0.062567 0.046852 0.002177 0...,6.701,54.760,0.996,7.878,2915.104,5.560,11.518,5.773
2,2023-10-17 08:32:21,04,运行,Normal,DE_径向,-3.299194\n-2.734741\n1.837646\n2.126831\n1.42...,"[749.9742, 642.3865, 642.5657, 750.4116, 750.3...",1.262,-0.118,-0.145,...,-0.253,0.003128 0.010294 0.008334 0.003554 0.002398 0...,7.703,82.435,0.992,8.268,4033.351,19.103,99.626,14.205
3,2023-10-17 16:32:24,04,运行,Normal,DE_轴向,-1.274902\n-0.9530029\n-0.5905762\n0.000732421...,"[416.4268, 352.1215, 381.6927, 572.6375, 352.5...",1.275,-0.015,-0.146,...,-0.219,0.002742 0.010710 0.027205 0.019433 0.007800 0...,7.729,74.168,0.997,6.796,2975.827,4.345,8.108,6.156
4,2023-10-17 16:32:24,04,运行,Normal,DE_径向,-2.385132\n-1.286133\n0.05334473\n1.00061\n1.7...,"[416.4268, 352.1215, 381.6927, 572.6375, 352.5...",1.329,-0.106,-0.199,...,-0.219,0.006931 0.017067 0.020207 0.007631 0.002921 0...,7.713,84.415,0.995,7.821,3933.128,5.344,11.277,7.809


In [30]:
df_final=df_final_all[[ 'turbine_id', 'data_category',
       'sensor_loc_std', 'acceleration',  'acc_std', 'acc_skew',
       'acc_kurt', 'rpm_mean', 'rpm_std', 'rpm_peak', 'rpm_skew', 'rpm_kurt',
       'spectral_kurtosis_acc_fft', 'crest_factor_acc_fft',
       'energy_high_ratio_acc_fft', 'spectral_entropy_acc_fft',
       'spectral_centroid_acc_fft', 'max_kurtosis', 'env_kurtosis',
       'env_crest_factor']]

In [31]:
df_final

,turbine_id,data_category,sensor_loc_std,acceleration,acc_std,acc_skew,acc_kurt,rpm_mean,rpm_std,rpm_peak,rpm_skew,rpm_kurt,spectral_kurtosis_acc_fft,crest_factor_acc_fft,energy_high_ratio_acc_fft,spectral_entropy_acc_fft,spectral_centroid_acc_fft,max_kurtosis,env_kurtosis,env_crest_factor
0,04,Normal,NDE_径向,-0.8442383\n0.7069092\n0.8236084\n0.9979248\n0...,1.778,0.002,-0.590,693.569,69.589,753.503,-0.840,-0.253,8.595,115.167,0.995,6.617,3220.209,4.794,6.826,4.428
1,04,Normal,DE_轴向,0.9194336\n1.142822\n0.9880371\n0.4608154\n-0....,1.376,-0.013,-0.067,693.569,69.589,753.503,-0.840,-0.253,6.701,54.760,0.996,7.878,2915.104,5.560,11.518,5.773
2,04,Normal,DE_径向,-3.299194\n-2.734741\n1.837646\n2.126831\n1.42...,1.262,-0.118,-0.145,693.569,69.589,753.503,-0.840,-0.253,7.703,82.435,0.992,8.268,4033.351,19.103,99.626,14.205
3,04,Normal,DE_轴向,-1.274902\n-0.9530029\n-0.5905762\n0.000732421...,1.275,-0.015,-0.146,511.070,96.705,761.611,0.224,-0.219,7.729,74.168,0.997,6.796,2975.827,4.345,8.108,6.156
4,04,Normal,DE_径向,-2.385132\n-1.286133\n0.05334473\n1.00061\n1.7...,1.329,-0.106,-0.199,511.070,96.705,761.611,0.224,-0.219,7.713,84.415,0.995,7.821,3933.128,5.344,11.277,7.809
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1314,28,Fault,NDE_径向,-1.449951\n-0.510498\n-0.927002\n-0.6717529\n-...,6.932,-0.288,3.715,1087.414,8.692,1108.292,0.343,-0.399,3.514,19.511,0.990,13.116,5549.700,9.530,16.616,6.545
1315,28,Fault,DE_径向,1.024292\n2.715576\n3.046143\n0.9029541\n0.565...,2.732,0.166,0.424,1087.414,8.692,1108.292,0.343,-0.399,6.455,44.247,0.870,10.886,2878.117,5.647,10.402,5.175
1316,28,Fault,NDE_径向,0.4580078\n0.935791\n-7.524048\n-4.655029\n2.6...,7.446,-0.143,2.826,1101.145,5.466,1113.090,-0.166,-0.600,2.912,12.331,0.997,13.457,5531.418,10.946,16.489,7.277
1317,28,Fault,DE_径向,-1.483887\n0.6230469\n0.7521973\n-0.18396\n2.6...,2.828,-0.050,0.265,1101.145,5.466,1113.090,-0.166,-0.600,5.807,37.212,0.921,11.830,3389.927,5.317,8.631,4.752


In [32]:
import os
import pandas as pd

# 1. 定义路径
save_dir = r"E:\\Grad\\data\\关键中转数据"
file_path = os.path.join(save_dir, "df_final.pkl")  # 后缀改为 .pkl

# 2. 确保文件夹存在
os.makedirs(save_dir, exist_ok=True)

# 3. 保存为 Pickle (protocol=5 是 Python 3.8+ 的高效协议，适合大数据)
df_final.to_pickle(file_path, protocol=5)

print(f"✅ [Pickle] 成功保存至：{file_path}")
print(f"📦 文件大小：{os.path.getsize(file_path) / (1024*1024):.2f} MB")

import pandas as pd

file_path = r"E:\\Grad\\data\\关键中转数据\\df_final.pkl"
df_final = pd.read_pickle(file_path)

display(df_final.head())

✅ [Pickle] 成功保存至：E:\\Grad\\data\\关键中转数据\df_final.pkl
📦 文件大小：1258.85 MB


,turbine_id,data_category,sensor_loc_std,acceleration,acc_std,acc_skew,acc_kurt,rpm_mean,rpm_std,rpm_peak,rpm_skew,rpm_kurt,spectral_kurtosis_acc_fft,crest_factor_acc_fft,energy_high_ratio_acc_fft,spectral_entropy_acc_fft,spectral_centroid_acc_fft,max_kurtosis,env_kurtosis,env_crest_factor
0,04,Normal,NDE_径向,-0.8442383\n0.7069092\n0.8236084\n0.9979248\n0...,1.778,0.002,-0.590,693.569,69.589,753.503,-0.840,-0.253,8.595,115.167,0.995,6.617,3220.209,4.794,6.826,4.428
1,04,Normal,DE_轴向,0.9194336\n1.142822\n0.9880371\n0.4608154\n-0....,1.376,-0.013,-0.067,693.569,69.589,753.503,-0.840,-0.253,6.701,54.760,0.996,7.878,2915.104,5.560,11.518,5.773
2,04,Normal,DE_径向,-3.299194\n-2.734741\n1.837646\n2.126831\n1.42...,1.262,-0.118,-0.145,693.569,69.589,753.503,-0.840,-0.253,7.703,82.435,0.992,8.268,4033.351,19.103,99.626,14.205
3,04,Normal,DE_轴向,-1.274902\n-0.9530029\n-0.5905762\n0.000732421...,1.275,-0.015,-0.146,511.070,96.705,761.611,0.224,-0.219,7.729,74.168,0.997,6.796,2975.827,4.345,8.108,6.156
4,04,Normal,DE_径向,-2.385132\n-1.286133\n0.05334473\n1.00061\n1.7...,1.329,-0.106,-0.199,511.070,96.705,761.611,0.224,-0.219,7.713,84.415,0.995,7.821,3933.128,5.344,11.277,7.809
